In [3]:
import pandas as pd
import nltk
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report
import re, string

nltk.download('stopwords')
nltk.download('wordnet')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Load the dataset
df = pd.read_csv("spam.csv", encoding="latin-1")
df = df[['v1', 'v2']]
df.columns = ['label', 'message']

print("Data loaded! Total rows:", len(df))
print(df.head())

Data loaded! Total rows: 5572
  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Era\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Era\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [4]:
stop_words = set(stopwords.words('english'))
lemm = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', ' ', text) # remove URLs
    text = re.sub(r'\S+@\S+', ' ', text)         # remove emails
    text = re.sub(r'\d+', ' ', text)             # remove numbers
    text = text.translate(str.maketrans('', '', string.punctuation)) # remove punctuation
    tokens = [lemm.lemmatize(w) for w in text.split() if w not in stop_words]
    return ' '.join(tokens)

# Remove duplicates and clean the messages
df = df.drop_duplicates()
df['clean_message'] = df['message'].apply(clean_text)
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

print("Text cleaned successfully!")
print(df[['message', 'clean_message']].head())

Text cleaned successfully!
                                             message  \
0  Go until jurong point, crazy.. Available only ...   
1                      Ok lar... Joking wif u oni...   
2  Free entry in 2 a wkly comp to win FA Cup fina...   
3  U dun say so early hor... U c already then say...   
4  Nah I don't think he goes to usf, he lives aro...   

                                       clean_message  
0  go jurong point crazy available bugis n great ...  
1                            ok lar joking wif u oni  
2  free entry wkly comp win fa cup final tkts st ...  
3                u dun say early hor u c already say  
4           nah dont think go usf life around though  


In [5]:
X = df['clean_message']
y = df['label']

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

# Build the model
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2))),
    ('svm', LinearSVC())
])

# Train the model
pipeline.fit(X_train, y_train)

# Test the model
pred = pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

Accuracy: 0.9845261121856866
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       903
           1       0.98      0.89      0.94       131

    accuracy                           0.98      1034
   macro avg       0.98      0.95      0.96      1034
weighted avg       0.98      0.98      0.98      1034



In [6]:
def predict_sms(text):
    cleaned = clean_text(text)
    prediction = pipeline.predict([cleaned])[0]
    return "Spam 🚨" if prediction == 1 else "Ham ✅"

# Test some messages
print(predict_sms("Congratulations! You have won a free iPhone. Click here now."))
print(predict_sms("Hi, are we meeting tomorrow at 3 PM?"))
print(predict_sms("URGENT! Your mobile number has been awarded a £2000 prize."))

Spam 🚨
Ham ✅
Spam 🚨


In [7]:
import joblib

# Save the trained model
joblib.dump(pipeline, 'spam_svm_model.pkl')
print("Model saved successfully!")

Model saved successfully!


In [8]:
# Create a list of custom messages to test
test_messages = [
    "Hey, are we still on for lunch tomorrow?",
    "URGENT! You have won a 1-week free membership to our gym. Call now!",
    "Your verification code is 998822. Do not share it with anyone.",
    "Congratulations! You've been selected for a $1000 Walmart gift card. Click here to claim.",
    "Mom, I'll be home late tonight, don't wait up.",
    "FREE entry into our weekly competition to win FA Cup final tickets! Text WIN to 80086",
    "Can you please send me the notes from class today?",
    "You have 1 new voicemail. Call 09061701461 to listen.",
]

# Loop through them and print the results
print("--- Testing Custom Messages ---\n")
for msg in test_messages:
    result = predict_sms(msg)
    print(f"{result} | Message: {msg}")

--- Testing Custom Messages ---

Ham ✅ | Message: Hey, are we still on for lunch tomorrow?
Ham ✅ | Message: URGENT! You have won a 1-week free membership to our gym. Call now!
Ham ✅ | Message: Your verification code is 998822. Do not share it with anyone.
Spam 🚨 | Message: Congratulations! You've been selected for a $1000 Walmart gift card. Click here to claim.
Ham ✅ | Message: Mom, I'll be home late tonight, don't wait up.
Spam 🚨 | Message: FREE entry into our weekly competition to win FA Cup final tickets! Text WIN to 80086
Ham ✅ | Message: Can you please send me the notes from class today?
Spam 🚨 | Message: You have 1 new voicemail. Call 09061701461 to listen.
